<a href="https://colab.research.google.com/github/noorarora/ARTI6000-RLHF-Assignment/blob/main/assignment1_rlhf/notebooks/01_sft_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Supervised Fine-Tuning (SFT) Baseline

This notebook trains a baseline instruction-following language model using the Alpaca dataset and the `distilgpt2` model.

## Objective
The goal of this stage is to create a baseline supervised fine-tuned (SFT) model that can later be compared against a preference-aligned model trained with RLHF techniques.

## Outputs
- Fine-tuned SFT model
- Saved tokenizer
- Example generated responses

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##  Install Dependencies

In [4]:
!pip install -q transformers datasets accelerate peft trl evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.3 MB/s eta 0:00:00


## Import Libraries and Set Random Seed

In [5]:
import os
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

print("Libraries loaded successfully")

Libraries loaded successfully


##  Load Base Model and Tokenizer

In [6]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Seed set successfully.")

Seed set successfully.


In [7]:
model_name = "distilgpt2"

print("Loading model:", model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded successfully")

Loading model: distilgpt2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully


## Load Instruction Dataset

In [8]:
dataset = load_dataset("tatsu-lab/alpaca")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})


In [9]:
dataset["train"][0]

{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}

## Format Training Examples

In [10]:
def format_example(example):
    instruction = example.get("instruction", "").strip()
    input_text = example.get("input", "").strip()
    output = example.get("output", "").strip()

    if input_text:
        text = f"Instruction: {instruction}\nInput: {input_text}\nResponse: {output}"
    else:
        text = f"Instruction: {instruction}\nResponse: {output}"

    return {"text": text}

formatted_dataset = dataset.map(format_example)
print(formatted_dataset["train"][0]["text"])

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

Instruction: Give three tips for staying healthy.
Response: 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.


In [11]:
def format_example(example):
    instruction = example.get("instruction", "").strip()
    input_text = example.get("input", "").strip()
    output = example.get("output", "").strip()

    if input_text:
        text = f"Instruction: {instruction}\nInput: {input_text}\nResponse: {output}"
    else:
        text = f"Instruction: {instruction}\nResponse: {output}"

    return {"text": text}

formatted_dataset = dataset.map(format_example)
print(formatted_dataset["train"][0]["text"])

Instruction: Give three tips for staying healthy.
Response: 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.


## Configure Training Arguments

In [12]:
small_train = formatted_dataset["train"].shuffle(seed=42).select(range(500))
print("Training subset size:", len(small_train))
print(small_train[0]["text"])

Training subset size: 500
Instruction: What would be the best type of exercise for a person who has arthritis?
Response: For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.


In [13]:
max_length = 256

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = small_train.map(tokenize_function)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

##  Train the SFT Model

In [14]:
tokenized_train = tokenized_train.remove_columns(
    [col for col in tokenized_train.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

print(tokenized_train[0])


{'input_ids': [6310, 2762, 25, 1867, 561, 307, 262, 1266, 2099, 286, 5517, 329, 257, 1048, 508, 468, 37954, 30, 198, 31077, 25, 1114, 2130, 351, 37954, 11, 262, 1266, 2099, 286, 5517, 561, 307, 1877, 12, 48240, 4568, 588, 20351, 11, 14899, 11, 393, 6155, 13, 2312, 13565, 2148, 262, 4034, 286, 5517, 1231, 22907, 803, 262, 7460, 286, 37954, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 5025

In [15]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator ready.")

Data collator ready.


##  Save the Fine-Tuned Model

In [16]:
output_dir = "/content/drive/MyDrive/rlhf_assignment/models/sft_baseline_distilgpt2"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=20,
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

print("Training arguments set.")

Training arguments set.


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator
)

print("Trainer created successfully.")

Trainer created successfully.


In [18]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,3.292347
40,2.898444
60,2.864199
80,2.818859
100,2.658665
120,2.748455
140,2.801281
160,2.664914
180,2.612094
200,2.694760


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=2.7925442962646483, metrics={'train_runtime': 47.6507, 'train_samples_per_second': 10.493, 'train_steps_per_second': 5.247, 'total_flos': 32662093824000.0, 'train_loss': 2.7925442962646483, 'epoch': 1.0})

In [19]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/rlhf_assignment/models/sft_baseline_distilgpt2/tokenizer_config.json',
 '/content/drive/MyDrive/rlhf_assignment/models/sft_baseline_distilgpt2/tokenizer.json')

In [20]:
print(f"Baseline SFT model saved to: {output_dir}")

Baseline SFT model saved to: /content/drive/MyDrive/rlhf_assignment/models/sft_baseline_distilgpt2


## Generate Example Responses

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

test_prompts = [
    "Instruction: Explain machine learning in simple terms.\nResponse:",
    "Instruction: Write a short paragraph about teamwork.\nResponse:",
    "Instruction: Give three tips for time management.\nResponse:"
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n" + "=" * 80)
    print(generated_text)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Instruction: Explain machine learning in simple terms.
Response: Machine learning is a powerful and powerful, powerful and efficient way to learn and to learn. It enables us to learn, learn and to learn and to learn. The most important part of the machine learning approach is to understand how to use it and learn more quickly, which can lead to more complex and complex learning models. Machine learning can be taught in the following programs:
Machine learning is a powerful


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Instruction: Write a short paragraph about teamwork.
Response: The common goal is to build good teamwork and teamwork. The most common tasks are managing tasks and creating good collaboration. The most common tasks are finding a team and meeting goals. The most common tasks are finding a team and meeting goals. The most common tasks are managing tasks and creating good collaboration. The most common tasks are finding a team and meeting goals. The most common tasks are finding a team and

Instruction: Give three tips for time management.
Response: Don’t get bogged down.
Response: Try to get things done quickly and quickly.
Input: Set up a schedule and plan for the next day.
Response: Start a week's workweek by making sure to keep an eye on the schedule and schedule.
Start a week's workweek by making sure to keep an eye on the schedule and schedule.
Start a
